## Citibike data ingestion
#### landing-bronze-silver-gold
#### by Matias Bertuzzi

##1. Configuration

In [0]:
import requests
from datetime import datetime, timezone
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType, DoubleType)
import pyspark.sql.functions as F

In [0]:
dbutils.widgets.text("catalog", "mbertuzzi", "Catalog")
dbutils.widgets.text("schema", "citibike", "Schema")
dbutils.widgets.text("landing_table", "landing_citibike_tripdata", "Landing:")
dbutils.widgets.text("bronze_table", "bronze_citibike_tripdata", "Bronze:")

In [0]:
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
landing_table = dbutils.widgets.get("landing_table")
bronze_table = dbutils.widgets.get("bronze_table")

full_table_name = f"{catalog}.{schema}.{landing_table}"
bronze_table = f"{catalog}.{schema}.{bronze_table}"
print(f"Reading from: {full_table_name}")
print(f"Writing to : {bronze_table}")

In [0]:
df = spark.read.table(full_table_name)

##2. Transform

In [0]:
df = df.withColumn("audit_date", F.current_timestamp())

##3. Write

In [0]:
count = df.count()

(
    df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(bronze_table)
)
 
print(f"Appended {count} rows to {bronze_table} at {datetime.now(timezone.utc)}")